In [2]:
%pip install qiskit torch numpy pandas

  Using cached qiskit-2.0.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached torch-2.7.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached numpy-2.2.6-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pandas-2.2.3-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached rustworkx-0.16.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
  Using cached scipy-1.15.3-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.4.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached typing_extensions-4.13.2-py3-none-any.whl.metadata (3.0 kB)
  Using cached symengine-0.13.0-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached filelock-3.1

In [6]:
import torch
import torch.nn as nn
from torch.autograd import Function
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

# … [data loading & preprocessing as before] …

# Qiskit StatevectorEstimator primitive
estimator = StatevectorEstimator()
observables = [ SparsePauliOp("ZI") ]  # measure Z on qubit 0

n_qubits = 2
def create_vqa_circuit(input_data, weights):
    qc = QuantumCircuit(n_qubits)
    # --- data encoding
    qc.ry(float(input_data[0]), 0)
    qc.ry(float(input_data[1]), 1)
    # --- ansatz layer
    for i in range(n_qubits):
        qc.ry(float(weights[2*i]),   i)
        qc.rz(float(weights[2*i+1]), i)
    # --- entangle
    qc.cx(0, 1)
    return qc

class VQALayerFunction(Function):
    @staticmethod
    def forward(ctx, input_tensor, weights):
        # detach & to numpy
        x = input_tensor.detach().cpu().numpy()
        w = weights.detach().cpu().numpy()
        ctx.save_for_backward(input_tensor, weights)
        # build & run
        qc = create_vqa_circuit(x, w)
        job = estimator.run([(qc, observables)])
        ev = job.result()[0].data.evs[0]        # scalar
        return torch.tensor([ev], dtype=torch.float32)

    @staticmethod
    def backward(ctx, grad_output):
        x_tensor, w_tensor = ctx.saved_tensors
        x = x_tensor.detach().cpu().numpy()
        w = w_tensor.detach().cpu().numpy()
        shift = np.pi / 2

        grads = []
        # parameter-shift for each weight
        for i in range(len(w)):
            w_plus  = w.copy(); w_minus = w.copy()
            w_plus[i]  += shift
            w_minus[i] -= shift

            qc_p = create_vqa_circuit(x, w_plus)
            qc_m = create_vqa_circuit(x, w_minus)

            ev_p = estimator.run([(qc_p, observables)]).result()[0].data.evs[0]
            ev_m = estimator.run([(qc_m, observables)]).result()[0].data.evs[0]

            grads.append((ev_p - ev_m) / 2.0)

        grads = torch.tensor(grads, dtype=torch.float32, device=w_tensor.device)
        # grad_input = None (we won’t backprop into x), grad_weights = grad_output * ∂f/∂w
        return None, grad_output.view(-1)[0] * grads

class VQALayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(4))

    def forward(self, x):
        # apply the custom Function to each sample in the batch
        out = [ VQALayerFunction.apply(x[i], self.weights) for i in range(x.size(0)) ]
        return torch.stack(out).view(-1, 1)

class HybridModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.classical = nn.Linear(n_features, 2)
        self.quantum   = VQALayer()
        self.output    = nn.Linear(1, 1)

    def forward(self, x):
        x = torch.tanh(self.classical(x))
        x = self.quantum(x)
        x = torch.sigmoid(self.output(x))
        return x

# instantiate
model    = HybridModel(X_train.shape[1])
optimizer= torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn  = nn.BCELoss()

# training
for epoch in range(50):
    optimizer.zero_grad()
    preds = model(X_train)
    loss  = loss_fn(preds, Y_train)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        acc = ((preds > 0.5).float() == Y_train).float().mean()
    print(f"Epoch {epoch+1:02d} | Loss {loss.item():.4f} | Train Acc {acc.item()*100:.1f}%")

# testing loop
with torch.no_grad():
    test_preds = model(X_test)
    test_acc   = ((test_preds > 0.5).float() == Y_test).float().mean()
    print(f"→ Test Accuracy: {test_acc.item()*100:.2f}%")


KeyboardInterrupt: 